# VQA Dataset Preprocessing
## Tạo subset từ VQA v2.0 Dataset

Notebook này giúp tạo một tập con của VQA v2.0 dataset với số lượng ảnh giới hạn để dễ dàng thử nghiệm trên Google Colab.

### Các bước:
1. Mount Google Drive (nếu dùng Colab)
2. Tải VQA v2.0 dataset
3. Tạo subset với N ảnh đầu tiên
4. Lưu file JSON subset

## 1. Setup và Import Libraries

In [ ]:
import json
import os
from collections import defaultdict
from typing import Dict, List, Set

print("Libraries imported successfully!")

## 2. Mount Google Drive (chỉ cho Colab)

In [ ]:
# Uncomment nếu chạy trên Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

# Đặt đường dẫn đến thư mục data
DATA_DIR = 'data'  # Thay đổi nếu cần
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")

## 3. Download VQA v2.0 Dataset (nếu chưa có)

Tải từ: https://visualqa.org/download.html

Cần tải:
- Training Questions
- Training Annotations
- Training Images (COCO 2014)

In [ ]:
# Tải dataset nếu chưa có (uncomment để sử dụng)
# !wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip -P {DATA_DIR}
# !wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip -P {DATA_DIR}
# !unzip {DATA_DIR}/v2_Questions_Train_mscoco.zip -d {DATA_DIR}
# !unzip {DATA_DIR}/v2_Annotations_Train_mscoco.zip -d {DATA_DIR}

print("Download completed (if executed)")

## 4. Functions để tạo Subset

In [ ]:
def load_json(file_path: str) -> dict:
    """Load JSON file."""
    print(f"Loading {file_path}...")
    with open(file_path, 'r') as f:
        return json.load(f)


def save_json(data: dict, file_path: str) -> None:
    """Save data to JSON file."""
    print(f"Saving to {file_path}...")
    with open(file_path, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Saved successfully!")


def create_subset(
    questions_file: str,
    annotations_file: str,
    output_questions_file: str,
    output_annotations_file: str,
    num_images: int = 5000
) -> None:
    """
    Create a subset of VQA dataset with the first N images.
    
    Args:
        questions_file: Path to original questions JSON file
        annotations_file: Path to original annotations JSON file
        output_questions_file: Path to save subset questions JSON
        output_annotations_file: Path to save subset annotations JSON
        num_images: Number of images to include in subset
    """
    # Load original data
    questions_data = load_json(questions_file)
    annotations_data = load_json(annotations_file)
    
    # Extract questions and annotations lists
    questions = questions_data.get('questions', [])
    annotations = annotations_data.get('annotations', [])
    
    print(f"\nOriginal dataset:")
    print(f"  Total questions: {len(questions)}")
    print(f"  Total annotations: {len(annotations)}")
    
    # Get unique image IDs from questions (maintaining order)
    image_ids_seen = []
    image_id_set: Set[int] = set()
    
    for q in questions:
        img_id = q['image_id']
        if img_id not in image_id_set:
            image_ids_seen.append(img_id)
            image_id_set.add(img_id)
            if len(image_ids_seen) >= num_images:
                break
    
    # Select first N unique images
    selected_image_ids = set(image_ids_seen[:num_images])
    
    print(f"\nSelected {len(selected_image_ids)} unique images")
    
    # Filter questions for selected images
    subset_questions = [
        q for q in questions 
        if q['image_id'] in selected_image_ids
    ]
    
    # Create question_id set for filtering annotations
    subset_question_ids = {q['question_id'] for q in subset_questions}
    
    # Filter annotations for selected questions
    subset_annotations = [
        ann for ann in annotations 
        if ann['question_id'] in subset_question_ids
    ]
    
    print(f"\nSubset dataset:")
    print(f"  Images: {len(selected_image_ids)}")
    print(f"  Questions: {len(subset_questions)}")
    print(f"  Annotations: {len(subset_annotations)}")
    
    # Create output data structure (preserve original format)
    output_questions_data = {
        'info': questions_data.get('info', {}),
        'task_type': questions_data.get('task_type', ''),
        'data_type': questions_data.get('data_type', ''),
        'data_subtype': questions_data.get('data_subtype', ''),
        'questions': subset_questions
    }
    
    output_annotations_data = {
        'info': annotations_data.get('info', {}),
        'license': annotations_data.get('license', {}),
        'data_type': annotations_data.get('data_type', ''),
        'data_subtype': annotations_data.get('data_subtype', ''),
        'annotations': subset_annotations
    }
    
    # Save subset files
    save_json(output_questions_data, output_questions_file)
    save_json(output_annotations_data, output_annotations_file)
    
    # Print statistics
    print(f"\n{'='*60}")
    print("Subset creation completed!")
    print(f"{'='*60}")
    print(f"Questions saved to: {output_questions_file}")
    print(f"Annotations saved to: {output_annotations_file}")

print("Functions defined successfully!")

## 5. Tạo Subset Dataset

In [ ]:
# Cấu hình
QUESTIONS_FILE = f"{DATA_DIR}/v2_OpenEnded_mscoco_train2014_questions.json"
ANNOTATIONS_FILE = f"{DATA_DIR}/v2_mscoco_train2014_annotations.json"
OUTPUT_QUESTIONS_FILE = f"{DATA_DIR}/subset_questions.json"
OUTPUT_ANNOTATIONS_FILE = f"{DATA_DIR}/subset_annotations.json"
NUM_IMAGES = 5000  # Số lượng ảnh muốn lấy

# Kiểm tra file tồn tại
if not os.path.exists(QUESTIONS_FILE):
    print(f"ERROR: File not found: {QUESTIONS_FILE}")
    print("Please download VQA v2.0 dataset first.")
elif not os.path.exists(ANNOTATIONS_FILE):
    print(f"ERROR: File not found: {ANNOTATIONS_FILE}")
    print("Please download VQA v2.0 dataset first.")
else:
    # Tạo subset
    create_subset(
        questions_file=QUESTIONS_FILE,
        annotations_file=ANNOTATIONS_FILE,
        output_questions_file=OUTPUT_QUESTIONS_FILE,
        output_annotations_file=OUTPUT_ANNOTATIONS_FILE,
        num_images=NUM_IMAGES
    )

## 6. Kiểm tra kết quả

In [ ]:
# Load và hiển thị một số mẫu từ subset
if os.path.exists(OUTPUT_QUESTIONS_FILE):
    subset_questions = load_json(OUTPUT_QUESTIONS_FILE)
    print("\nSample questions from subset:")
    for i, q in enumerate(subset_questions['questions'][:5]):
        print(f"\n{i+1}. Image ID: {q['image_id']}")
        print(f"   Question: {q['question']}")
        print(f"   Question ID: {q['question_id']}")

## Hoàn tất!

Bây giờ bạn đã có:
- `subset_questions.json` - Questions cho subset
- `subset_annotations.json` - Annotations cho subset

Có thể sử dụng các file này với notebook training tiếp theo.